# Weekly Assignment III — Questions 2 to 5

This notebook provides worked solutions (code + explanations) for Questions 2 to 5 from the provided PDF. Each question is contained in its own section with runnable code. You can run the notebook cells in order.

**Notes:**
- Some cells use `scikit-learn` for convenience; please ensure `scikit-learn`, `pandas`, `numpy`, and `matplotlib` are installed in your environment.
- For Question 4 (Banknote dataset) the code attempts to download the dataset from the UCI repository. If your environment does not have internet access, download the dataset manually and put it in the working directory as `data_banknote_authentication.txt`.

---


## Question 2 — Decision Tree (from scratch) using Gini Index

**Task:** Construct a Decision Tree classifier from scratch using the Gini Index as splitting criterion on the Borrower dataset (Figure 1). The implementation below:
- computes Gini for candidate splits,
- chooses the best split,
- recursively builds a tree until pure or no split possible,
- supports numeric and categorical attributes.

We use the Borrower dataset from Question 1 as input.


In [ ]:

import pandas as pd
import numpy as np
from pprint import pprint

# Borrower dataset
data = {
    'Home Owner': ['Yes','No','No','Yes','No','No','Yes','No','No','No'],
    'Marital Status': ['Single','Married','Single','Married','Divorced','Married','Divorced','Single','Married','Single'],
    'Annual Income': [125,100,70,120,95,60,220,85,75,90],
    'Defaulted Borrower': ['No','No','No','No','Yes','Yes','No','Yes','No','Yes']
}
df = pd.DataFrame(data)

# Utility functions for Gini-based decision tree
def gini(groups, classes):
    n_instances = float(sum([len(group) for group in groups]))
    if n_instances == 0:
        return 0
    gini_value = 0.0
    for group in groups:
        size = float(len(group))
        if size == 0:
            continue
        score = 0.0
        for class_val in classes:
            p = (group['Defaulted Borrower'] == class_val).sum() / size
            score += p * p
        gini_value += (1.0 - score) * (size / n_instances)
    return gini_value

def test_split_numeric(column, value, dataset):
    left = dataset[dataset[column] <= value]
    right = dataset[dataset[column] > value]
    return left, right

def test_split_categorical(column, val, dataset):
    left = dataset[dataset[column] == val]
    right = dataset[dataset[column] != val]
    return left, right

def get_split(dataset, features):
    classes = dataset['Defaulted Borrower'].unique()
    b_index, b_value, b_score, b_groups, b_type = None, None, float('inf'), None, None

    for col in features:
        if dataset[col].dtype.kind in 'bO' or dataset[col].dtype == object:
            # categorical: try each unique value
            for val in dataset[col].unique():
                groups = test_split_categorical(col, val, dataset)
                g = gini(groups, classes)
                if g < b_score:
                    b_index, b_value, b_score, b_groups, b_type = col, val, g, groups, 'cat'
        else:
            # numeric: try splitting at midpoints between sorted values
            unique_vals = sorted(dataset[col].unique())
            for i in range(len(unique_vals)-1):
                val = (unique_vals[i] + unique_vals[i+1]) / 2.0
                groups = test_split_numeric(col, val, dataset)
                g = gini(groups, classes)
                if g < b_score:
                    b_index, b_value, b_score, b_groups, b_type = col, val, g, groups, 'num'
    return {'index': b_index, 'value': b_value, 'groups': b_groups, 'type': b_type, 'gini': b_score}

def to_terminal(group):
    # return the most common class label in group
    return group['Defaulted Borrower'].mode().iloc[0]

def build_tree(node, max_depth, min_size, depth, features):
    left, right = node['groups']
    del(node['groups'])
    # check for no split
    if left.empty or right.empty:
        combined = pd.concat([left, right])
        node['left'] = node['right'] = to_terminal(combined)
        return
    # check for max depth
    if depth >= max_depth:
        node['left'], node['right'] = to_terminal(left), to_terminal(right)
        return
    # left child
    if len(left) <= min_size:
        node['left'] = to_terminal(left)
    else:
        node['left'] = get_split(left, features)
        build_tree(node['left'], max_depth, min_size, depth+1, features)
    # right child
    if len(right) <= min_size:
        node['right'] = to_terminal(right)
    else:
        node['right'] = get_split(right, features)
        build_tree(node['right'], max_depth, min_size, depth+1, features)

def fit_train(dataset, max_depth=5, min_size=1):
    features = [c for c in dataset.columns if c != 'Defaulted Borrower']
    root = get_split(dataset, features)
    build_tree(root, max_depth, min_size, 1, features)
    return root

def predict(node, row):
    if isinstance(node, str):
        return node
    if node['type'] == 'cat':
        if row[node['index']] == node['value']:
            return predict(node['left'], row)
        else:
            return predict(node['right'], row)
    else:
        if row[node['index']] <= node['value']:
            return predict(node['left'], row)
        else:
            return predict(node['right'], row)

# Build tree and show
tree = fit_train(df, max_depth=5, min_size=1)
print('Constructed tree (nested dict):')
pprint(tree)

# Test predictions on training set
preds = df.apply(lambda r: predict(tree, r), axis=1)
df_result = df.copy()
df_result['predicted'] = preds
print('\nPredictions on training set:')
display(df_result)
print('\nAccuracy on training data:', (df_result['predicted'] == df_result['Defaulted Borrower']).mean())


---

## Question 3 — C4.5 rules, coverage & accuracy (Play dataset)

**Task:** Using the dataset illustrated via Figure 2 (the classic Play dataset / Weather dataset),
- generate C4.5-style rules,
- compute coverage and accuracy of each rule,
- compute overall model performance (actual vs predicted dataframe).

**Note:** Implementing full C4.5 (gain ratio, continuous handling, pruning) is large; here we:
- train a decision tree using entropy criterion (ID3-like) as an approximation to C4.5,
- extract rules from the trained tree,
- compute coverage and accuracy per extracted rule.


In [ ]:

# Play (Weather) dataset - classic example
play_data = {
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool','Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
    'Humidity': ['High','High','High','High','Normal','Normal','Normal','High','Normal','Normal','Normal','High','Normal','High'],
    'Windy': [False, True, False, False, False, True, True, False, False, False, True, True, False, True],
    'Play': ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No']
}
p_df = pd.DataFrame(play_data)
display(p_df)


In [ ]:

from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn import tree
import numpy as np

# Prepare data (encode categorical features)
X = p_df[['Outlook','Temperature','Humidity','Windy']].copy()
X_enc = pd.get_dummies(X, drop_first=False)  # keep all dummy columns to map back
y = p_df['Play']

# Train a tree (entropy)
clf = DecisionTreeClassifier(criterion='entropy', random_state=0)
clf.fit(X_enc, y)

# Function to extract rules
def tree_to_rules(clf, feature_names):
    tree_ = clf.tree_
    feature_name = [
        feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!"
        for i in tree_.feature
    ]

    paths = []

    def recurse(node, path):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            # For one-hot encoded features, threshold will be 0.5 to separate 0/1
            # Build left and right conditions
            left_path = path + [f"({name} <= {threshold:.3f})"]
            recurse(tree_.children_left[node], left_path)
            right_path = path + [f"({name} > {threshold:.3f})"]
            recurse(tree_.children_right[node], right_path)
        else:
            # leaf
            vals = tree_.value[node][0]
            total = vals.sum()
            probs = vals/total
            cls = clf.classes_[np.argmax(probs)]
            paths.append((path, cls, probs.max(), int(total)))
    recurse(0, [])
    return paths

feature_names = X_enc.columns.tolist()
rules = tree_to_rules(clf, feature_names)
print('Extracted rules (conditions over one-hot features) and predicted class:')
for conds, cls, conf, samples in rules:
    print('\nRule ->', ' AND '.join(conds))
    print('Predict =>', cls, '| confidence:', conf, '| covered samples (leaf count):', samples)

# Map rules back to original features (best-effort): compute coverage and accuracy per rule
def rule_matches(conds, X_enc):
    # evaluate conditions on X_enc to get boolean mask
    mask = np.ones(len(X_enc), dtype=bool)
    for c in conds:
        # c looks like "(Feature <= 0.5)" or "(Feature > 0.5)"
        feat, op, val = c.strip('()').split()
        val = float(val)
        if op == '<=':
            mask &= (X_enc[feat] <= val)
        else:
            mask &= (X_enc[feat] > val)
    return mask

results = []
for conds, cls, conf, _ in rules:
    mask = rule_matches(conds, X_enc)
    covered = p_df[mask]
    if len(covered) == 0:
        coverage = 0.0
        acc = None
    else:
        coverage = len(covered) / len(p_df)
        acc = (covered['Play'] == cls).mean()
    results.append({'conditions': conds, 'predicted': cls, 'coverage': coverage, 'accuracy': acc, 'covered_count': len(covered)})

print('\nRule coverage & accuracy:')
for r in results:
    print(r)

# Overall model performance (actual vs predicted)
p_df_pred = p_df.copy()
p_df_pred_enc = X_enc.copy()
p_df_pred['predicted'] = clf.predict(X_enc)
display(p_df_pred)
from sklearn.metrics import accuracy_score, classification_report
print('\nOverall accuracy:', accuracy_score(p_df_pred['Play'], p_df_pred['predicted']))
print('\nClassification report:\n', classification_report(p_df_pred['Play'], p_df_pred['predicted']))


---

## Question 4 — K-Nearest Neighbors (Banknote Authentication)

**Task:** Predict whether the bank note is authentic using KNN. We implement a standard KNN using scikit-learn and evaluate accuracy using cross-validation.

**Note:** The dataset is available at the UCI repository. The code will attempt to download it; if your environment is offline, place the file `data_banknote_authentication.txt` in the working directory (same format as UCI dataset).


In [ ]:

# Question 4: KNN on Banknote Authentication dataset
import os
import urllib.request

data_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt'
local_file = 'data_banknote_authentication.txt'

# try to download if not present
if not os.path.exists(local_file):
    try:
        print('Downloading dataset from UCI...')
        urllib.request.urlretrieve(data_url, local_file)
        print('Download complete.')
    except Exception as e:
        print('Could not download dataset automatically. Please download manually and save as', local_file)
        print('Error:', e)

# load dataset
cols = ['variance','skewness','curtosis','entropy','class']
df_bank = pd.read_csv(local_file, header=None, names=cols)
display(df_bank.head())

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

X = df_bank[['variance','skewness','curtosis','entropy']].values
y = df_bank['class'].values

# scale features
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(Xs, y, test_size=0.25, random_state=0, stratify=y)

# fit KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print('Test set accuracy:', knn.score(X_test, y_test))

# cross-validation accuracy
cv_scores = cross_val_score(knn, Xs, y, cv=5)
print('5-fold cross-validation accuracies:', cv_scores)
print('Mean CV accuracy:', cv_scores.mean())


----

## Question 5 — Naive Bayes Classifier (probability of Play when Outlook=Overcast and Temperature=Mild)

**Task:** Implement a Naive Bayes classifier (categorical) to calculate the probability of `Play=Yes/No` given `Outlook=Overcast` and `Temperature=Mild`. We'll use the Play dataset from Question 3 and compute conditional probabilities using the Naive Bayes assumption (features independent given class). We'll use Laplace smoothing for stability.


In [ ]:

# Using the same play dataset p_df from Q3 (if you restarted kernel, recreate it)
try:
    p_df
except NameError:
    play_data = {
        'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain','Overcast','Sunny','Sunny','Rain','Sunny','Overcast','Overcast','Rain'],
        'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool','Cool','Mild','Cool','Mild','Mild','Mild','Hot','Mild'],
        'Humidity': ['High','High','High','High','Normal','Normal','Normal','High','Normal','Normal','Normal','High','Normal','High'],
        'Windy': [False, True, False, False, False, True, True, False, False, False, True, True, False, True],
        'Play': ['No','No','Yes','Yes','Yes','No','Yes','No','Yes','Yes','Yes','Yes','Yes','No']
    }
    p_df = pd.DataFrame(play_data)

# Function for categorical naive bayes probabilities with Laplace smoothing
def categorical_naive_bayes(df, feature_vals, target_col='Play', alpha=1):
    classes = df[target_col].unique()
    prior = {}
    likelihood = {}  # nested dict: likelihood[class][feature][value] = P(value|class)
    total_count = len(df)
    for c in classes:
        subset = df[df[target_col] == c]
        prior[c] = (len(subset) + alpha) / (total_count + alpha*len(classes))
        likelihood[c] = {}
        for feat, val in feature_vals.items():
            # count occurrences of val in feat among class c
            counts = subset[feat].value_counts().to_dict()
            # number of distinct possible values for this feature in whole df
            distinct = df[feat].nunique()
            count_val = counts.get(val, 0)
            # Laplace smoothing
            likelihood[c].setdefault(feat, {})[val] = (count_val + alpha) / (len(subset) + alpha*distinct)
    return prior, likelihood

# Query: Outlook=Overcast, Temperature=Mild
query = {'Outlook': 'Overcast', 'Temperature': 'Mild'}

prior, likelihood = categorical_naive_bayes(p_df, query, target_col='Play', alpha=1)
print('Priors:', prior)
print('\nLikelihoods P(feature=value | class):')
for cls in likelihood:
    print('\nClass:', cls)
    for feat in likelihood[cls]:
        print(feat, '->', query.get(feat), ':', likelihood[cls][feat][query.get(feat)])

# Compute posterior probabilities (unnormalized) and normalize
posterior = {}
for cls in prior:
    post = prior[cls]
    for feat, val in query.items():
        post *= likelihood[cls][feat][val]
    posterior[cls] = post

# normalize
total_post = sum(posterior.values())
posterior_norm = {k: v/total_post for k, v in posterior.items()}
print('\nPosterior probabilities given the evidence (Outlook=Overcast, Temperature=Mild):')
print(posterior_norm)
print('\nPredicted Class:', max(posterior_norm, key=posterior_norm.get))


----

### Final notes
- The provided implementations are educational and intentionally explicit to show the algorithms.
- For production, prefer robust libraries (scikit-learn) for performance and edge-case handling.

Feel free to ask for any cell-by-cell explanation or for separated notebook files per question.
